# Del 1

In [ ]:
import time
import random

def S(U):
    L = range(len(U))
    return sum([ x**2 for x in L ])

def gen_nbit_rand(n):
    return random.getrandbits(n)

def gen_nbit_rand_odd(n):
    return random.getrandbits(n) | 1

class Hash(object):
    @staticmethod
    def mulshift(a=0, l=0):
        def H(x):
            return (a * x) >> (64 -l)
        return H


    @staticmethod
    def mulmodprime(a=0, b=0, p=0, l=0):
        q = 89

        def f(x):
            y = (x & p) + (x >> q)
            if y >= p:
                y -= p
            return y

        def H(x):
            y = f(x * a)
            y += b
            y = f(y)
            return y & ((2 ** l) - 1)
        return H


class HashTable(object):
    def __init__(self, l, h=Hash.mulmodprime) -> None:
        self.h = h
        self.size = 2**l
        self.table = [[]] * self.size

    def __getitem__(self, key):
        for k, v in self.table[self.h(key)]:
            if k == key:
                return v
        raise KeyError(key)

    def __setitem__(self, key, value):
        for i, (k, _) in enumerate(self.table[self.h(key)]):
            if k == key:
                self.table[self.h(key)][i] = (key, value)
                return
        self.table[self.h(key)].append((key, value))

    def __len__(self):
        return sum(len(bucket) for bucket in self.table)

    def __delitem__(self, key):
        bucket = self.table[self.h(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                del bucket[i]
                return
        raise KeyError(key)

    def increment(self, key, d):
        for i, (k, v) in enumerate(self.table[self.h(key)]):
            if k == key:
                self.table[self.h(key)][i] = (k, int(v) + d)
                return
        self.table[self.h(key)].append((key, d))

if __name__ == "__main__":
    import create_stream


    gen_nbit_rand(256)

    l = 15
    n = 2**20
    p = 2**89 - 1
    a1 = 0b0111010010101101000111010010111100001011001110100011101100001111
    a2 = 0b0111010010101101000111010010111100001011001110100011101100001110 
    b = 0b1000111011010111000110101111000100001011001110001010110111101010
    sum_mulshift_hash = 0
    sum_mulmodprime_hash = 0
    
    time_mulshift_hash = 0
    time_mulmodprime_hash = 0

    test_stream = create_stream.CreateStream(n, l)
    
    time_mulmodprime_hash = time.time()
    for x, _ in test_stream:
        sum_mulmodprime_hash += Hash.mulmodprime(a2, b, p, l)(x) ** 2  # Calculate square and add to sum
    time_taken_mulmodprime_hash = time.time() - time_mulmodprime_hash

    # Measure time for mulshift hash
    time_mulshift_hash = time.time()
    for x, _ in test_stream:
        sum_mulshift_hash += Hash.mulshift(a1, l)(x) ** 2  # Calculate square and add to sum
    time_taken_mulshift_hash = time.time() - time_mulshift_hash


    print(f"Time Taken mulmodprime: {time_taken_mulmodprime_hash:.7f}")
    print(f"Time Taken mulshift: {time_taken_mulshift_hash:.7f}")


    # Opgave 3
     
    new_l = 20
    while True:
        try:
            t = HashTable(new_l, Hash.mulmodprime(a2, b, p, l))
        except MemoryError:
            print("MemError L: ", new_l)
            new_l -= 1
            continue
        except OverflowError:
            print("OverflowError L: ", new_l)
            new_l -= 1
            continue
        break
    print("Found L", new_l)

    # new_l = 64
    # while True:
    #     try:
    #         t = HashTable(new_l, Hash.mulshift(a1, l))
    #     except MemoryError:
    #         print("MemError L: ", new_l)
    #         new_l -= 1
    #         continue
    #     except OverflowError:
    #         print("OverflowError L: ", new_l)
    #         new_l -= 1
    #         continue
    #     break
    # print("Found L", new_l)



# Del 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from create_stream import CreateStream

b = 89 # (89 / 8) = 12 bytes
m = 2**64 #k = m = 2t

a0 = 0x478ad369f6852eac0de7ccf7
a1 = 0xd472f7b830cf473f771bf810
a2 = 0x695b4252f2c52ba0031649c4
a3 = 0xd4e0ab8bb4907f43b1ede881
A = [a0,a1,a2,a3]

#Opgave 4
def q_universal_hash(b, A, x):
    p = (2**b)-1 #0x01FFFFFFFFFFFFFFFFFFFFFF
    q = len(A)
    a = list(map(lambda x: p&x, A)) # this ensures each coefficient is within the range [0,p−1]

    y = a[q-1] 
    for i in range(q-2, -1, -1): # starts with the highest degree coefficient
        y = y*x + a[i] # multiplies by x and adds the next coefficient
        y = (y&p) + (y>>b)  
    if (y >= p):  
        y = y-p 
    return y

#Opgave 5
def sign_and_hash(k, b, A, x):
    #q_universal_hash(x) mod k
    g = q_universal_hash(b, A, x) 
    h = g & (k - 1)
    b = g >> (b - 1)
    s = 1 - (2*b)
    return (s,h)

#Opgave 6
class CountSketch:
    def __init__(self, t, b, A):
        self.t = t
        self.b = b
        self.A = A
        self.m = 2**t
        self.table = np.zeros(self.m, dtype=int) # Initialize a table with m zeros
    
    def update(self, x, s):
        s, h = sign_and_hash(self.m, self.b, self.A, x)
        self.table[h] += s # Update the table at position h by adding s
    
    def estimate(self):
        return np.sum(self.table**2) # Estimate by summing squares of the table entries

def CountSketchTest():
    # Example
    t = 16  # Example t value (log2(m))
    count_sketch = CountSketch(t, b, A)

    # Generate stream and update sketch
    stream = CreateStream(10000, 16)

    for x, s in stream:
        count_sketch.update(x, s)

    # Estimate the sum of squared counts
    estimate = count_sketch.estimate()
    print("Estimate:", estimate)

#Opgave 7 
def hashing_parameters(path):
    #Generate parameters for 100 random 4-universal hashfunctions:
    file = open(path,'r') ##4800 byte in hex from https://www.random.org/bytes/
    content = file.read().split()
    AA = []
    for i in range(0,len(content),12*4):
        A = []
        for j in range(0,12*4,12):
            hex_string = ""
            for k in range(12):
                hex_string += content[i+j+k]
            A.append(int(hex_string, 16))
        AA.append(A)
    return AA

def plot(n, l, t, b, S, path):
    AA = hashing_parameters(path)
    X = []
    for i in range(0,len(AA),1):
        stream = CreateStream(n, l)
        count_sketch = CountSketch(t, b, AA[i])
        for x, s in stream:
            count_sketch.update(x, s)
        X.append(count_sketch.estimate())
    Xx = list(range(1,101))
    Xy = sorted(X, key=lambda x: x, reverse=False)
    plt.plot(Xx, Xy, '.')

    M = []
    for i in range(0,99,11):
        G = []
        for j in range(0,11):
            G.append(X[i+j])
        G.sort(key=lambda x: x, reverse=False)
        M.append(G[5])
    Mx = list(range(6,100,11))
    My = sorted(M, key=lambda x: x, reverse=False)
    plt.plot(Mx, My, '.')
    
    plt.plot([1,100], [S,S])

    plt.show()        
            
plot(10000, 16, 16, b, 10000, 'RandomHex.txt')